# Lab 2 — Cost Controls & Budget Middleware

**Goal:** Hard-stop spending before it gets out of hand. Every production agent needs this.

**Real incident:** In 2023 several startups received $10,000+ OpenAI bills from a single runaway agent that looped overnight. A budget middleware is the $10 solution to a $10,000 problem.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from middleware import BudgetMiddleware, BudgetExceededError
from openai import OpenAI

client = OpenAI()

def base_agent(prompt: str, **kwargs) -> str:
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=100,
    )
    return resp.choices[0].message.content

print('Ready ✓')

## 1. Global budget — stops after limit

In [ ]:
# Tiny budget for demo purposes
budget_agent = BudgetMiddleware(base_agent, global_budget_usd=0.005, per_user_budget_usd=0.005)

queries = ['What is 2+2?', 'Explain Docker in one sentence.', 'What is FastAPI?', 'Tell me about Python.']

for i, q in enumerate(queries):
    try:
        result = budget_agent(q, user_id='user1')
        print(f'Query {i+1}: {result[:80]}')
        print(f'  Spent so far: ${budget_agent._global_spent:.5f}')
    except BudgetExceededError as e:
        print(f'Query {i+1}: BUDGET EXCEEDED — {e}')
        break

## 2. Per-user isolation

In [ ]:
# User A has a tiny budget, User B has more
multi_user_agent = BudgetMiddleware(
    base_agent,
    global_budget_usd=1.0,
    per_user_budget_usd=0.003,
)

users = [('userA', 'What is Python?'), ('userA', 'What is Docker?'), ('userB', 'What is Python?')]

for user_id, query in users:
    try:
        result = multi_user_agent(query, user_id=user_id)
        spent = multi_user_agent._user_spent[user_id]
        print(f'[{user_id}] OK — spent ${spent:.5f}')
    except BudgetExceededError as e:
        print(f'[{user_id}] BLOCKED: {e}')

print('\nFull spending report:')
import json
print(json.dumps(multi_user_agent.spending_report(), indent=2))

## 3. Cost attribution dashboard

In [ ]:
report = multi_user_agent.spending_report()
print(f'Global: ${report["global_spent_usd"]:.5f} / ${report["global_budget_usd"]} ({report["global_spent_usd"]/report["global_budget_usd"]:.1%} used)')
print('\nPer-user:')
for user, spent in report['users'].items():
    pct = spent / 0.003 * 100
    bar = '█' * int(pct / 5)
    print(f'  {user:10} ${spent:.5f} [{bar:<20}] {pct:.0f}%')

## Exercise
Extend `BudgetMiddleware` to support **rolling 24-hour windows**:  
- Instead of a lifetime budget, track spend in the last 24 hours
- After 24 hours from the first request, the window resets
- Add a `daily_budget_usd` parameter

**Hint:** Store timestamps alongside spend amounts and filter on `time.time() - 86400`.

In [ ]:
# Your code here
